# NC Capstone: A.I. Business Sentiment and Employment by State
This notebook loads and combines BLS and BTOS datasets using relative paths, then seeks to analyze the relationship between A.I. sentiment and employment by type and U.S. State over time. 

---

Load Libraries

In [1]:
import pandas as pd
from glob import glob
import os

Checking to Make Sure File Path is Working

In [2]:
bls_paths = sorted(glob("data/state_M20*_dl.xlsx"))
print("Files found:")
print(bls_paths)

Files found:
['data\\state_M2020_dl.xlsx', 'data\\state_M2021_dl.xlsx', 'data\\state_M2022_dl.xlsx', 'data\\state_M2023_dl.xlsx', 'data\\state_M2024_dl.xlsx']


Load and Combine BLS Data
This block loads all BLS Excel files from the `data/` folder.

In [3]:

bls_dfs = []

for path in bls_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        bls_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BLS data
bls_combined = pd.concat(bls_dfs, ignore_index=True)
print("\n BLS data combined. Sample:")
display(bls_combined.head())


 BLS data combined. Sample:


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,41.07,18690,24060,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,91.89,47740,67330,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,#,49480,97930,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,#,48030,67740,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,16220,17190,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN


Adding 'Tech' Classification to BLS Occupational Data

In [45]:
# Add tech classification column to BLS data
print("Adding tech classification to BLS data")

# Create the tech flag based on OCC_TITLE containing "Data" or "Software"
bls_combined['Tech_Classification'] = bls_combined['OCC_TITLE'].apply(
    lambda x: 'Tech' if pd.notna(x) and ('DATA' in str(x).upper() or 'SOFTWARE' in str(x).upper()) 
    else 'Non Tech'
)

# Verify the classification
tech_counts = bls_combined['Tech_Classification'].value_counts()
print(f"\nTech Classification Counts:")
print(tech_counts)

# Show some examples of Tech-classified jobs
print(f"\nSample Tech-classified occupations:")
tech_jobs = bls_combined[bls_combined['Tech_Classification'] == 'Tech']['OCC_TITLE'].unique()
print(tech_jobs[:10])  # Show first 10 unique tech job titles

print(f"\nBLS data updated with Tech_Classification column.")
print(f"New shape: {bls_combined.shape}")

Adding tech classification to BLS data

Tech Classification Counts:
Tech_Classification
Non Tech    184322
Tech          1432
Name: count, dtype: int64

Sample Tech-classified occupations:
['Database Administrators and Architects'
 'Software Developers and Software Quality Assurance Analysts and Testers'
 'Data Scientists and Mathematical Science Occupations, All Other'
 'Data Entry Keyers' 'Database Administrators' 'Database Architects'
 'Software Developers' 'Software Quality Assurance Analysts and Testers'
 'Data Scientists']

BLS data updated with Tech_Classification column.
New shape: (185754, 34)


BTOS Collection Date Translation Added Here

In [46]:
date_translation_path = "data/btos_collection_dates.csv"
try:
    date_translation = pd.read_csv(date_translation_path)
    print("BTOS date translation loaded successfully.")
    display(date_translation.head())
except Exception as e:
    print(f"Failed to load {date_translation_path}: {e}")

BTOS date translation loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


Load and Combine BTOS Data.  
This block loads the BTOS survey data files from the `data/` folder.

In [47]:
btos_paths = ["data/State.xlsx", "data/State_v1.xlsx"]
btos_dfs = []

for path in btos_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        btos_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BTOS data
btos_combined = pd.concat(btos_dfs, ignore_index=True)
print("\n BTOS data combined. Sample:")
display(btos_combined.head())


 BTOS data combined. Sample:


,State,Question ID,Question,Answer ID,Answer,202512,202511,202510,202509,202508,...,202224,202223,202222,202221,202220,202219,202218,202217,202216,202215
0,AK,2.0,"Overall, how would you describe this business'...",1.0,Excellent,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,2.0,"Overall, how would you describe this business'...",2.0,Above average,17.9%,16.5%,13.9%,19.6%,21.4%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,2.0,"Overall, how would you describe this business'...",3.0,Average,46.2%,52.9%,69.6%,34.6%,57.1%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,2.0,"Overall, how would you describe this business'...",4.0,Below average,18.2%,18.2%,S,32.1%,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,2.0,"Overall, how would you describe this business'...",5.0,Poor,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Renaming BTOS Date Columns to Make them More Usable

In [48]:

collection_dates = pd.read_csv(date_translation_path)
print("BTOS collection dates loaded successfully.")
display(collection_dates.head())

# Convert Smpdt to string and create mapping
collection_dates['Smpdt'] = collection_dates['Smpdt'].astype(str)
date_mapping = dict(zip(collection_dates['Smpdt'], collection_dates['Ref End']))

# Check current BTOS columns
print("Current BTOS date columns:", [col for col in btos_combined.columns if str(col).startswith('202')][:10])

# Find matching columns and rename
date_columns = [col for col in btos_combined.columns if str(col) in date_mapping.keys()]
print(f"Found {len(date_columns)} matching columns to rename")

if len(date_columns) > 0:
    rename_dict = {col: date_mapping[str(col)] for col in date_columns}
    btos_combined = btos_combined.rename(columns=rename_dict)
    print(f"Successfully renamed {len(rename_dict)} date columns to Ref End dates.")
    
    # Verify the renaming worked
    print("Sample new column names:", list(btos_combined.columns)[-10:])
else:
    print("No matching columns found")

BTOS collection dates loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


Current BTOS date columns: ['202512', '202511', '202510', '202509', '202508', '202507', '202506', '202505', '202504', '202503']
Found 76 matching columns to rename
Successfully renamed 76 date columns to Ref End dates.
Sample new column names: ['11/20/2022', '11/06/2022', '10/23/2022', '10/09/2022', '09/25/2022', '09/11/2022', '08/28/2022', '08/14/2022', '07/31/2022', '07/17/2022']


Adding 'Region' Column to Both Files

In [52]:
region_mapping = {
    # Northeast
    'ME': 'Northeast', 'NH': 'Northeast', 'VT': 'Northeast', 'MA': 'Northeast', 
    'RI': 'Northeast', 'CT': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast', 
    'NJ': 'Northeast', 'DE': 'Northeast', 'MD': 'Northeast',
    
    # Southeast
    'FL': 'Southeast', 'AL': 'Southeast', 'GA': 'Southeast', 'SC': 'Southeast', 
    'NC': 'Southeast', 'VA': 'Southeast', 'WV': 'Southeast', 'KY': 'Southeast', 
    'TN': 'Southeast', 'MS': 'Southeast', 'AR': 'Southeast', 'LA': 'Southeast',
    
    # Midwest
    'OH': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'IL': 'Midwest', 
    'WI': 'Midwest', 'MN': 'Midwest', 'IA': 'Midwest', 'MO': 'Midwest', 
    'ND': 'Midwest', 'SD': 'Midwest', 'NE': 'Midwest', 'KS': 'Midwest',
    
    # Southwest
    'TX': 'Southwest', 'NM': 'Southwest', 'AZ': 'Southwest', 'OK': 'Southwest',
    
    # West
    'CA': 'West', 'OR': 'West', 'WA': 'West', 'NV': 'West', 'ID': 'West', 
    'MT': 'West', 'WY': 'West', 'UT': 'West', 'CO': 'West', 'AK': 'West', 'HI': 'West',

    # Territories
    'DC': 'Territories', 'GU': 'Territories', 'PR': 'Territories', 'VI': 'Territories',
    'XX': 'Territories'
}

# Add Region column to BLS data
print("Adding regional classification to BLS data...")
bls_combined['Region'] = bls_combined['PRIM_STATE'].map(region_mapping)

# Add Region column to BTOS data  
print("Adding regional classification to BTOS data...")
btos_combined['Region'] = btos_combined['State'].map(region_mapping)

# Verify the regional classifications
print(f"\nBLS Regional Distribution:")
print(bls_combined['Region'].value_counts())

print(f"\nBTOS Regional Distribution:")
print(btos_combined['Region'].value_counts())

# Check for any unmapped states
bls_unmapped = bls_combined[bls_combined['Region'].isna()]['PRIM_STATE'].unique()
btos_unmapped = btos_combined[btos_combined['Region'].isna()]['State'].unique()

if len(bls_unmapped) > 0:
    print(f"\nUnmapped BLS states: {bls_unmapped}")
if len(btos_unmapped) > 0:
    print(f"\nUnmapped BTOS states: {btos_unmapped}")

print(f"\nRegional classification complete!")
print(f"BLS data shape: {bls_combined.shape}")
print(f"BTOS data shape: {btos_combined.shape}")

Adding regional classification to BLS data...
Adding regional classification to BTOS data...

BLS Regional Distribution:
Region
Southeast      44633
Midwest        43745
Northeast      37951
West           36975
Southwest      14799
Territories     7651
Name: count, dtype: int64

BTOS Regional Distribution:
Region
Southeast      2160
Midwest        2160
West           1980
Northeast      1980
Southwest       720
Territories     455
Name: count, dtype: int64

Unmapped BTOS states: [nan
 'Source: U.S. Census Bureau, Business Trends and Outlook Survey (BTOS). The Census Bureau has reviewed this data product to ensure appropriate access, use, and disclosure avoidance protection of the confidential source data.\n        (Project No. P-7529868, Disclosure Review Board (DRB) approval number: CBDRB-FY24-0474)'
 'Source: U.S. Census Bureau, Business Trends and Outlook Survey (BTOS), Posted date: September 14, 2023, Project No. EID-7529868 / Approval CBDRB-FY22-341.']

Regional classification co

Creating a Year Column for BLS Data

In [57]:
# Extract year from BLS source_file column
print("Extracting year from BLS source files...")

# Extract 4-digit year from filenames like "state_M2020_dl.xlsx"
bls_combined['Year'] = bls_combined['source_file'].str.extract(r'(\d{4})')

# Convert to integer for easier analysis
bls_combined['Year'] = bls_combined['Year'].astype(int)

# Verify the year extraction
print(f"\nYear extraction verification:")
year_counts = bls_combined['Year'].value_counts().sort_index()
print(year_counts)

# Show sample of source files and extracted years
print(f"\nSample source files and extracted years:")
sample_check = bls_combined[['source_file', 'Year']].drop_duplicates().sort_values('Year')
print(sample_check)

print(f"\nYear column successfully added to BLS data!")
print(f"Years covered: {bls_combined['Year'].min()} - {bls_combined['Year'].max()}")

Extracting year from BLS source files...

Year extraction verification:
Year
2020    36085
2021    37580
2022    37569
2023    37676
2024    36844
Name: count, dtype: int64

Sample source files and extracted years:
                source_file  Year
0       state_M2020_dl.xlsx  2020
36085   state_M2021_dl.xlsx  2021
73665   state_M2022_dl.xlsx  2022
111234  state_M2023_dl.xlsx  2023
148910  state_M2024_dl.xlsx  2024

Year column successfully added to BLS data!
Years covered: 2020 - 2024


Save Combined Outputs

In [59]:
bls_combined.to_csv("data/combined_bls.csv", index=False)
btos_combined.to_csv("data/combined_btos.csv", index=False)
print("\n Output saved: data/combined_bls.csv and data/combined_btos.csv")


 Output saved: data/combined_bls.csv and data/combined_btos.csv


Combined Staging Data Preview

In [60]:
print("\nPreview: BLS Combined Data")
display(bls_combined.head())

print("\nPreview: BTOS Combined Data")
display(btos_combined.head())


Preview: BLS Combined Data


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT,Tech_Classification,Region,Year
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech,Southeast,2020
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech,Southeast,2020
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech,Southeast,2020
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech,Southeast,2020
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN,Non Tech,Southeast,2020



Preview: BTOS Combined Data


,State,Question_ID,Question,Answer_ID,Answer,06/01/2025,05/18/2025,05/04/2025,04/20/2025,04/06/2025,...,11/06/2022,10/23/2022,10/09/2022,09/25/2022,09/11/2022,08/28/2022,08/14/2022,07/31/2022,07/17/2022,Region
16,AK,7.0,"In the last two weeks, did this business use A...",1.0,Yes,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West
17,AK,7.0,"In the last two weeks, did this business use A...",2.0,No,86.5%,90.6%,86.8%,92.6%,94.5%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West
18,AK,7.0,"In the last two weeks, did this business use A...",3.0,Do not know,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West
92,AK,26.0,"During the next six months, do you think this ...",1.0,Yes,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West
93,AK,26.0,"During the next six months, do you think this ...",2.0,No,57.2%,74.5%,77.8%,74.1%,78.6%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West


Selecting the Relevant A.I. Questions from the BTOS Survey

In [55]:
from pandasql import sqldf

btos_combined.columns = btos_combined.columns.str.replace(' ', '_')
btos_combined = btos_combined.loc[:, ~btos_combined.columns.isna()]

# I am only using the "State" source - which goes as far back as Sept 2023, 
# (cont'd) because prior to Sept 2023 (State v1 file) there was not an A.I. question in the data.

query = """
SELECT distinct Question, Question_ID
FROM btos_combined
WHERE 1=1
and Question like '%Artificial Intelligence%'
and source_file in ('State.xlsx')
limit 10
"""
print("\nQuerying BTOS Combined Data for AI-related questions:")


result = sqldf(query, locals())


print(result)


Querying BTOS Combined Data for AI-related questions:
                                            Question  Question_ID
0  In the last two weeks, did this business use A...          7.0
1  During the next six months, do you think this ...         26.0


Getting Cleaned BTOS Dataset Together

In [56]:
# Filter BTOS data to only AI-related questions
print("Filtering BTOS data to AI-related questions only")

# Filter for only the two AI questions we identified
ai_questions_filter = (btos_combined['Question_ID'].isin([7.0, 26.0])) & \
                     (btos_combined['source_file'] == 'State.xlsx')

btos_ai_only = btos_combined[ai_questions_filter].copy()

print(f"Original BTOS data shape: {btos_combined.shape}")
print(f"Filtered AI-only data shape: {btos_ai_only.shape}")
print(f"Reduction: {btos_combined.shape[0] - btos_ai_only.shape[0]:,} rows removed")

# Verify the filtering worked
print("\nUnique questions remaining:")
verification_query = """
SELECT DISTINCT Question_ID, Question, COUNT(*) as row_count
FROM btos_ai_only
GROUP BY Question_ID, Question
"""
verification_result = sqldf(verification_query, locals())
print(verification_result)

# Update the main dataframe reference
btos_combined = btos_ai_only
print("\nBTOS dataset successfully reduced to AI-related questions only.")

Filtering BTOS data to AI-related questions only
Original BTOS data shape: (9459, 83)
Filtered AI-only data shape: (318, 83)
Reduction: 9,141 rows removed

Unique questions remaining:
   Question_ID                                           Question  row_count
0          7.0  In the last two weeks, did this business use A...        159
1         26.0  During the next six months, do you think this ...        159

BTOS dataset successfully reduced to AI-related questions only.


Custom Functions - 1, getting current and perceived near term A.I. adoption rates

In [90]:
import pandas as pd
from typing import List

def get_btos_yes_rates(df: pd.DataFrame, date_columns: List[str]) -> pd.DataFrame:

    results = []
    
    # Get unique states and questions
    states = df['State'].unique()
    questions = df['Question_ID'].unique()
    
    for state in states:
        if pd.isna(state):
            continue
            
        # Get region for this state
        state_data = df[df['State'] == state]
        region = state_data['Region'].iloc[0] if not state_data.empty and 'Region' in state_data.columns else None
            
        for question_id in questions:
            if pd.isna(question_id):
                continue
                
            # Get question text for clarity
            question_text = df[df['Question_ID'] == question_id]['Question'].iloc[0] if len(df[df['Question_ID'] == question_id]) > 0 else f"Question {question_id}"
            
            for date_col in date_columns:
                if date_col not in df.columns:
                    continue
                
                # Filter for this state and question
                state_question_data = df[
                    (df['State'] == state) & 
                    (df['Question_ID'] == question_id)
                ]
                
                if state_question_data.empty:
                    continue
                
                # Get the 'Yes' response for this date
                yes_data = state_question_data[state_question_data['Answer'] == 'Yes']
                
                if not yes_data.empty and date_col in yes_data.columns:
                    yes_value = yes_data[date_col].iloc[0]
                    
                    # Convert percentage to number
                    if pd.notna(yes_value) and yes_value != 'S':
                        try:
                            if isinstance(yes_value, str) and '%' in yes_value:
                                yes_rate = float(yes_value.replace('%', ''))
                            else:
                                yes_rate = float(yes_value)
                                
                            results.append({
                                'State': state,
                                'Region': region,
                                'Question_ID': question_id,
                                'Question_Type': 'Current_Usage' if question_id == 7.0 else 'Future_Plans',
                                'Date': date_col,
                                'Yes_Rate': yes_rate,
                                'Question_Text': question_text[:50] + '...'  # Truncated for readability
                            })
                        except (ValueError, TypeError):
                            continue
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort for better readability
    if not results_df.empty:
        results_df = results_df.sort_values(['State', 'Question_ID', 'Date'])
    
    return results_df

Region Usage

In [91]:
# Get AI Yes rates with regions
target_dates = ['12/29/2024', '12/31/2023']
yes_rates_df = get_btos_yes_rates(btos_combined, target_dates)

# Regional analysis
regional_summary = yes_rates_df.groupby(['Region', 'Question_Type', 'Date'])['Yes_Rate'].mean().round(2)
print("Average AI Yes Rates by Region:")
print(regional_summary)

Average AI Yes Rates by Region:
Region       Question_Type  Date      
Midwest      Current_Usage  12/29/2024     5.51
                            12/31/2023     3.77
             Future_Plans   12/29/2024     8.76
                            12/31/2023     5.54
Northeast    Current_Usage  12/29/2024     5.42
                            12/31/2023     4.18
             Future_Plans   12/29/2024     9.11
                            12/31/2023     6.53
Southeast    Current_Usage  12/29/2024     5.67
                            12/31/2023     4.40
             Future_Plans   12/29/2024     8.74
                            12/31/2023     5.66
Southwest    Current_Usage  12/29/2024     6.03
                            12/31/2023     4.95
             Future_Plans   12/29/2024     7.32
                            12/31/2023     5.70
Territories  Current_Usage  12/29/2024     8.70
                            12/31/2023     5.15
             Future_Plans   12/29/2024    11.30
                 

State Usage

In [92]:
# Just current AI usage (Question 7)
current_usage = yes_rates_df[yes_rates_df['Question_ID'] == 7.0]
print("Current AI Usage Yes Rates:")
print(current_usage[['State', 'Date', 'Yes_Rate']])

# Just future AI plans (Question 26)  
future_plans = yes_rates_df[yes_rates_df['Question_ID'] == 26.0]
print("Future AI Plans Yes Rates:")
print(future_plans[['State', 'Date', 'Yes_Rate']])

Current AI Usage Yes Rates:
    State        Date  Yes_Rate
0      AL  12/29/2024       3.5
1      AL  12/31/2023       3.3
4      AR  12/29/2024       5.8
6      AZ  12/29/2024       6.1
7      AZ  12/31/2023       5.3
..    ...         ...       ...
141    WA  12/31/2023       5.9
144    WI  12/29/2024       4.8
145    WI  12/31/2023       2.7
148    XX  12/29/2024       8.7
149    XX  12/31/2023       5.0

[70 rows x 3 columns]
Future AI Plans Yes Rates:
    State        Date  Yes_Rate
2      AL  12/29/2024       6.5
3      AL  12/31/2023       4.8
5      AR  12/29/2024       9.0
8      AZ  12/29/2024       7.9
9      AZ  12/31/2023       7.3
..    ...         ...       ...
143    WA  12/31/2023       7.4
146    WI  12/29/2024       8.9
147    WI  12/31/2023       5.1
150    XX  12/29/2024      13.8
151    XX  12/31/2023       9.4

[82 rows x 3 columns]


Customer Function 2 - Tech Employment Movement by State During the Same Period

In [93]:
import pandas as pd
from typing import List

def get_tech_employment_summary(df: pd.DataFrame, years: List[int]) -> pd.DataFrame:

    # Filter for specified years
    filtered_df = df[df['Year'].isin(years)].copy()
    
    # Clean TOT_EMP column - convert to numeric, handle special characters
    filtered_df['TOT_EMP_Clean'] = pd.to_numeric(filtered_df['TOT_EMP'], errors='coerce')
    
    # Group by State, Region, Year, Tech_Classification and sum TOT_EMP
    summary = filtered_df.groupby(['PRIM_STATE', 'Region', 'Year', 'Tech_Classification'])['TOT_EMP_Clean'].sum().reset_index()
    
    # Pivot to get Tech and Non Tech as separate columns
    pivot_summary = summary.pivot_table(
        index=['PRIM_STATE', 'Region', 'Year'], 
        columns='Tech_Classification', 
        values='TOT_EMP_Clean', 
        fill_value=0
    ).reset_index()
    
    # Flatten column names
    pivot_summary.columns.name = None
    
    # Calculate totals and percentages
    pivot_summary['Total_Employment'] = pivot_summary['Non Tech'] + pivot_summary['Tech']
    pivot_summary['Tech_Percentage'] = round((pivot_summary['Tech'] / pivot_summary['Total_Employment'] * 100), 2)
    
    # Rename columns for clarity
    pivot_summary = pivot_summary.rename(columns={
        'PRIM_STATE': 'State',
        'Non Tech': 'Non_Tech_Employment',
        'Tech': 'Tech_Employment'
    })
    
    # Convert employment numbers to integers for cleaner display
    pivot_summary['Tech_Employment'] = pivot_summary['Tech_Employment'].astype(int)
    pivot_summary['Non_Tech_Employment'] = pivot_summary['Non_Tech_Employment'].astype(int)
    pivot_summary['Total_Employment'] = pivot_summary['Total_Employment'].astype(int)
    
    # Sort by region and state
    pivot_summary = pivot_summary.sort_values(['Region', 'State', 'Year'])
    
    return pivot_summary

Tech Employment by State / Region

In [94]:
# Get tech employment using actual TOT_EMP numbers
tech_summary = get_tech_employment_summary(bls_combined, [2023, 2024])
print(tech_summary.head())

   State   Region  Year  Non_Tech_Employment  Tech_Employment  \
25    IA  Midwest  2023              4615900            15180   
26    IA  Midwest  2024              4664480            14820   
29    IL  Midwest  2023             17903740            82960   
30    IL  Midwest  2024             18042890            79110   
31    IN  Midwest  2023              9443590            22700   

    Total_Employment  Tech_Percentage  
25           4631080             0.33  
26           4679300             0.32  
29          17986700             0.46  
30          18122000             0.44  
31           9466290             0.24  


Custom Function 3 - Uniting BLS and BOTS Data by State and Region

In [95]:
import pandas as pd
from typing import List

def unite_tech_and_ai_data(bls_df: pd.DataFrame, btos_df: pd.DataFrame, 
                          years: List[int], dates: List[str]) -> pd.DataFrame:

    # Get tech employment data (keeps separate rows for each year)
    tech_data = get_tech_employment_summary(bls_df, years)
    
    # Get AI adoption data
    ai_data = get_btos_yes_rates(btos_df, dates)
    
    # Map dates to years for merging
    date_to_year = {}
    for date in dates:
        year = int(date.split('/')[-1])  # Extract year from MM/DD/YYYY format
        date_to_year[date] = year
    
    # Add year column to AI data based on the 'Date' column from BTOS function
    ai_data['Year'] = ai_data['Date'].map(date_to_year)
    
    # Average AI data by State, Region, Year, and Question_Type (in case multiple dates per year)
    ai_avg = ai_data.groupby(['State', 'Region', 'Year', 'Question_Type']).agg({
        'Yes_Rate': 'mean'
    }).round(2).reset_index()
    
    # Pivot AI data to get Current_Usage and Future_Plans as separate columns
    ai_pivot = ai_avg.pivot_table(
        index=['State', 'Region', 'Year'], 
        columns='Question_Type', 
        values='Yes_Rate', 
        fill_value=0
    ).reset_index()
    
    # Flatten column names
    ai_pivot.columns.name = None
    
    # Rename AI columns for clarity
    if 'Current_Usage' in ai_pivot.columns:
        ai_pivot = ai_pivot.rename(columns={'Current_Usage': 'AI_Current_Usage'})
    if 'Future_Plans' in ai_pivot.columns:
        ai_pivot = ai_pivot.rename(columns={'Future_Plans': 'AI_Future_Plans'})
    
    # Merge tech and AI data on State, Region, and Year
    combined_data = pd.merge(tech_data, ai_pivot, on=['State', 'Region', 'Year'], how='left')
    
    # Add some calculated fields for easier plotting
    combined_data['Tech_Employment_Thousands'] = round(combined_data['Tech_Employment'] / 1000, 1)
    combined_data['Total_Employment_Millions'] = round(combined_data['Total_Employment'] / 1000000, 2)
    
    # Fill any missing AI data with 0 (states that might not have AI survey data)
    combined_data['AI_Current_Usage'] = combined_data['AI_Current_Usage'].fillna(0)
    combined_data['AI_Future_Plans'] = combined_data['AI_Future_Plans'].fillna(0)
    
    # Sort by region, state, and year for better readability
    combined_data = combined_data.sort_values(['Region', 'State', 'Year'])
    
    return combined_data

Results in a Table

In [96]:
plot_data = unite_tech_and_ai_data(
    bls_combined, 
    btos_combined, 
    [2023, 2024], 
    ['12/31/2023', '12/29/2024']
)

print("Combined dataset created!")
print(f"Shape: {plot_data.shape}")
print("\nFirst 10 rows:")
print(plot_data.head(10))

# Check Alabama 2023 specifically
al_2023 = plot_data[(plot_data['State'] == 'AL') & (plot_data['Year'] == 2023)]
print(f"\nAlabama 2023:")
print(al_2023[['State', 'Year', 'AI_Current_Usage', 'AI_Future_Plans']])

Combined dataset created!
Shape: (107, 11)

First 10 rows:
  State   Region  Year  Non_Tech_Employment  Tech_Employment  \
0    IA  Midwest  2023              4615900            15180   
1    IA  Midwest  2024              4664480            14820   
2    IL  Midwest  2023             17903740            82960   
3    IL  Midwest  2024             18042890            79110   
4    IN  Midwest  2023              9443590            22700   
5    IN  Midwest  2024              9528380            21980   
6    KS  Midwest  2023              4204800            18300   
7    KS  Midwest  2024              4268670            19310   
8    MI  Midwest  2023             12977000            57940   
9    MI  Midwest  2024             13103260            56760   

   Total_Employment  Tech_Percentage  AI_Current_Usage  AI_Future_Plans  \
0           4631080             0.33               2.8              4.8   
1           4679300             0.32               7.5              8.5   
2          

Alabama Test

In [97]:
# Get all Alabama data (both years)
al_data = plot_data[plot_data['State'] == 'AL']
print("Alabama - Complete Data:")
print(al_data)

# Show Alabama trends
print(f"\nAlabama Trends (2023 → 2024):")
print(f"Tech Employment: {al_data[al_data['Year']==2023]['Tech_Employment'].iloc[0]:,} → {al_data[al_data['Year']==2024]['Tech_Employment'].iloc[0]:,}")
print(f"Tech Percentage: {al_data[al_data['Year']==2023]['Tech_Percentage'].iloc[0]}% → {al_data[al_data['Year']==2024]['Tech_Percentage'].iloc[0]}%")
print(f"AI Current Usage: {al_data[al_data['Year']==2023]['AI_Current_Usage'].iloc[0]}% → {al_data[al_data['Year']==2024]['AI_Current_Usage'].iloc[0]}%")
print(f"AI Future Plans: {al_data[al_data['Year']==2023]['AI_Future_Plans'].iloc[0]}% → {al_data[al_data['Year']==2024]['AI_Future_Plans'].iloc[0]}%")

Alabama - Complete Data:
   State     Region  Year  Non_Tech_Employment  Tech_Employment  \
46    AL  Southeast  2023              6113920            23030   
47    AL  Southeast  2024              6216930            24960   

    Total_Employment  Tech_Percentage  AI_Current_Usage  AI_Future_Plans  \
46           6136950             0.38               3.3              4.8   
47           6241890             0.40               3.5              6.5   

    Tech_Employment_Thousands  Total_Employment_Millions  
46                       23.0                       6.14  
47                       25.0                       6.24  

Alabama Trends (2023 → 2024):
Tech Employment: 23,030 → 24,960
Tech Percentage: 0.38% → 0.4%
AI Current Usage: 3.3% → 3.5%
AI Future Plans: 4.8% → 6.5%
